In [14]:
# OpenBLAS starts one thread per core even for tiny BLAS-1 calls, and the Krylov
# solvers below issue thousands of those per iteration. On a 24-core machine the
# oversubscription cost 4.7x. This has to be set *before* numpy is imported, so if
# numpy is already loaded, restart the kernel.
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import splu, cg, bicgstab, LinearOperator
%matplotlib qt5

In [15]:
# Inputs
# ------

# Immersed body (circular cylinder) and doubly-periodic domain
R  = 1
Lx = 10
Ly = 10
nx = 128
ny = 128

# Initial timestep (the time derivative acts as the SIMPLEC under-relaxation)
dt = 0.1

# Kinematic viscosity. VAR[IP] is the reduced pressure p/rho, so the density
# never appears explicitly in the equations below.
nu = 1/20

# Linear solvers. Both give the same answer; only the cost differs.
#   'direct' - SuperLU factorisation of both systems every iteration.
#   'krylov' - BiCGStab on the momentum, preconditioned by a factorisation reused
#              across iterations; CG on the pressure, preconditioned by a
#              constant-coefficient FFT Poisson solve.
# Measured on this machine (s per iteration, 10 iterations, one thread):
#     16k cells   direct 0.15   krylov 0.18
#     66k cells   direct 1.06   krylov 1.08      <- crossover
#    262k cells   direct 12.8   krylov 7.0
# The FFT preconditioner is much cheaper when nx and ny have small prime factors
# (161 = 7*23 costs about twice as much per FFT as 160).
linearSolver = 'direct'
pTol         = 1e-12       # CG relative tolerance for the pressure equation

In [16]:
# Grid, fields and sparsity pattern
# ---------------------------------
IU, IV, IP = 0, 1, 2                 # component indices into VAR

dx, dy = Lx/nx, Ly/ny
V      = dx*dy                       # cell volume
nCells = nx*ny

xp = (np.arange(nx)+0.5)*dx          # cell centres
yp = (np.arange(ny)+0.5)*dy
xf = np.arange(nx+1)*dx              # face centres
yf = np.arange(ny+1)*dy
XP, YP = np.meshgrid(xp, yp)
XF, YF = np.meshgrid(xf, yf)

VAR = np.zeros((3, ny, nx))          # u, v, p

# The 5-point stencil is held as five (ny,nx) coefficient arrays in the order
# P, E, W, N, S -- not as an (nCells,nCells) matrix. Its sparsity pattern never
# changes, so the CSR structure and the permutation from stencil order into CSR
# data order are built once, here. Assembling a matrix is then an O(nnz)
# fancy-index rather than an O(nCells^2) scan over a 99.92%-empty dense array.
ID  = np.arange(nCells).reshape(ny, nx)
ROW = np.stack([ID]*5, -1).ravel()
COL = np.stack([ID, np.roll(ID, -1, 1), np.roll(ID, 1, 1),
                    np.roll(ID, -1, 0), np.roll(ID, 1, 0)], -1).ravel()
assert len(set(zip(ROW.tolist(), COL.tolist()))) == ROW.size, \
    'stencil aliases itself: nx and ny must both be at least 3'

_pattern    = csr_matrix((np.arange(1.0, ROW.size+1), (ROW, COL)),
                         shape=(nCells, nCells))
CSR_PERM    = (_pattern.data-1).astype(np.int64)
CSR_INDICES = _pattern.indices
CSR_INDPTR  = _pattern.indptr


def zeroStencil():
    """An empty set of stencil coefficients, indexed [P|E|W|N|S, iy, ix]."""
    return np.zeros((5, ny, nx))


def toCSR(a):
    """Assemble stencil coefficients into a sparse matrix."""
    return csr_matrix((a.reshape(5, -1).T.ravel()[CSR_PERM], CSR_INDICES, CSR_INDPTR),
                      shape=(nCells, nCells))


def offDiagDot(a, phi):
    """(A - diag(aP)) @ phi, as four neighbour shifts. No matrix is formed."""
    return (a[1]*np.roll(phi, -1, 1) + a[2]*np.roll(phi, 1, 1)
          + a[3]*np.roll(phi, -1, 0) + a[4]*np.roll(phi, 1, 0))


def wrapX(f):
    """Cell-array -> face-array in x: append the periodic image of the first face."""
    return np.concatenate([f, f[:, :1]], axis=1)


def wrapY(f):
    return np.concatenate([f, f[:1, :]], axis=0)

## Discrete problem

Finite volumes on a uniform, doubly periodic, **collocated** Cartesian grid ($u$, $v$, $p$
all at cell centres), advanced with implicit Euler and split with the SIMPLEC algorithm
as in OpenFOAM's `simpleFoam`. $p$ is the reduced pressure $p/\rho$.

**Momentum predictor** (per component, $V=\Delta x \Delta y$):

$$ \underbrace{\frac{V}{\Delta t}u^{n+1}}_{\texttt{fvm\_ddt\_LHS}}
 + \underbrace{\nabla\!\cdot\!(\phi\,u^{n+1})}_{\texttt{fvm\_divConvFlux}}
 - \underbrace{\nabla\!\cdot\!(\nu\nabla u^{n+1})}_{\texttt{fvm\_lap}\,+\,\texttt{ibm\_lap}}
 \;=\; \underbrace{\frac{V}{\Delta t}u^{n}}_{\texttt{fvm\_ddt\_RHS}}
 + \underbrace{V g}_{\texttt{source}}
 - \underbrace{V \nabla p^{n}}_{\texttt{fvc\_grd}} $$

Written as $a_P u_P = H(u) - V\nabla p$ with $H(u) = b - \sum_{nb} a_{nb} u_{nb}$, so that
$\texttt{HbyA} = H(u)/a_P$ and $\phi_{HbyA} = \texttt{interpolatePhiIBM}(\texttt{HbyA})$.

**SIMPLEC** replaces $r_{A_U}=1/a_P$ by $r_{A_tU} = 1/(a_P + \sum_{nb} a_{nb})$, and solves

$$ \nabla\!\cdot\!\left(r_{A_tU}\,V \nabla p\right) = \nabla\!\cdot\!\phi_{HbyA}
\qquad\Longrightarrow\qquad
\phi = \phi_{HbyA} - r_{A_tU} V\,\mathrm{snGrad}(p)\,|S_f|, \quad
u = \texttt{HbyA} - r_{A_tU}\,V\nabla p .$$

## Immersed boundary

Following **Luchini, Gatti, Chiarini, Gattere, Atzori & Quadrio,
*J. Comput. Phys.* **539** (2025) 114245**, the boundary is accounted for by modifying
**only the centre weight of the laplacian stencil** (`ibm_lap`, their Eq. 6):
the ghost value in the solid is linearly extrapolated from the first fluid point and the
wall and substituted back, so nothing has to be stored at the ghost point and the
corrections from different directions are additive. No pressure or continuity correction
is introduced. Because the momentum equation is solved implicitly, the IBM term is
implicit in time by construction, so the explicit-scheme machinery of their Sec. 2.4
(Eqs. 14–17) is not needed here.

## Linear algebra

The 5-point stencil is stored as five $(n_y,n_x)$ **coefficient arrays**, never as a dense
$(N,N)$ matrix, and assembled through a sparsity pattern built once. The pressure
reference uses OpenFOAM's own `fvMatrix::setReference` (double the diagonal of the
reference cell), which — unlike replacing the row — keeps the matrix **symmetric** while
giving exactly the same solution, because the right-hand side is compatible.

Two interchangeable solver paths, selected by `linearSolver`; the *equations* are
identical, only their solution differs:

- `direct` — SuperLU on both systems.
- `krylov` — BiCGStab on the momentum, preconditioned by a factorisation **reused**
  across iterations; CG on the pressure, preconditioned by a **constant-coefficient FFT
  Poisson solve** inside a symmetric Jacobi scaling. Both are preconditioners only, so
  `pEqn.flux()` and the SIMPLEC splitting are untouched and the result matches `direct`
  to solver tolerance.

In [17]:
# Finite-volume operators
# -----------------------
# `fvm_*` add to the implicit stencil coefficients, `fvc_*` return explicit fields.
# Everything is doubly periodic, so every neighbour access is an np.roll.

def fvm_ddt_LHS(a, dt):
    """Implicit Euler d/dt: adds V/dt to the diagonal."""
    a[0] += V/dt


def fvm_ddt_RHS(rhs, phi, dt):
    """Explicit part of the implicit-Euler d/dt: V*phi^n/dt."""
    rhs += V*phi/dt


def fvm_lap(a, Gamma):
    """
    Implicit laplacian(Gamma, .) with linear face interpolation of Gamma.
    Sign convention: negative diagonal, positive off-diagonals.
    """
    gE = 0.5*(Gamma+np.roll(Gamma, -1, 1))*dy/dx
    gW = 0.5*(Gamma+np.roll(Gamma,  1, 1))*dy/dx
    gN = 0.5*(Gamma+np.roll(Gamma, -1, 0))*dx/dy
    gS = 0.5*(Gamma+np.roll(Gamma,  1, 0))*dx/dy
    a[0] -= gE+gW+gN+gS
    a[1] += gE
    a[2] += gW
    a[3] += gN
    a[4] += gS


def fvm_divConvFlux(a, phiX, phiY):
    """Implicit div(phi u) with central differencing."""
    fE, fW = phiX[:, 1:], phiX[:, :-1]
    fN, fS = phiY[1:, :], phiY[:-1, :]
    a[0] += 0.5*(fE-fW+fN-fS)
    a[1] += 0.5*fE
    a[2] -= 0.5*fW
    a[3] += 0.5*fN
    a[4] -= 0.5*fS


def fvc_grd(phi):
    """Volume-integrated Gauss gradient, sum_f phi_f Sf (that is, V*grad(phi))."""
    return np.stack([0.5*dy*(np.roll(phi, -1, 1)-np.roll(phi, 1, 1)),
                     0.5*dx*(np.roll(phi, -1, 0)-np.roll(phi, 1, 0))])


def fvc_grdFlux(phi, Gamma):
    """Face fluxes Gamma_f * snGrad(phi) * |Sf|, i.e. the flux of fvm_lap(Gamma, phi)."""
    fx = (phi-np.roll(phi, 1, 1))/dx * 0.5*(Gamma+np.roll(Gamma, 1, 1))*dy
    fy = (phi-np.roll(phi, 1, 0))/dy * 0.5*(Gamma+np.roll(Gamma, 1, 0))*dx
    return wrapX(fx), wrapY(fy)


def fvc_divFlux(phiX, phiY):
    """Volume-integrated divergence of a face-flux field."""
    return (phiX[:, 1:]-phiX[:, :-1]) + (phiY[1:, :]-phiY[:-1, :])

In [18]:
# Immersed boundary
# -----------------
IBM_SOLID_WEIGHT = 1e12    # penalty enforcing u = 0 in cells inside the body
BISECT_TOL       = 1e-12   # *relative* tolerance for locating the wall on an arm


def isInBody(x, y):
    return ((x-Lx/2)**2 + (y-Ly/2)**2) <= R**2


solid = isInBody(XP, YP)
fluid = ~solid


def wallDistance(sx, sy, x, y, h):
    """
    Distance in (0,h) from (x,y) to the body surface along the direction (sx,sy),
    by bisection on every point at once. The tolerance is relative to h, so
    refining the mesh refines the wall position with it.
    """
    inAt0 = isInBody(x, y)
    sIn   = np.where(inAt0, 0.0, h)
    sOut  = np.where(inAt0, h, 0.0)
    for _ in range(int(np.ceil(-np.log2(BISECT_TOL)))+1):
        sMid = 0.5*(sIn+sOut)
        hit  = isInBody(x+sx*sMid, y+sy*sMid)
        sIn  = np.where(hit, sMid, sIn)
        sOut = np.where(hit, sOut, sMid)
    return 0.5*(sIn+sOut)


def ibm_lap(a):
    """
    Immersed-boundary correction of the laplacian, Luchini et al. (2025), Eq. (6).

    The ghost value at the first point inside the solid is replaced by its linear
    extrapolation from the first fluid point and the wall (where the value is
    zero). Substituting it back folds the whole correction into the *centre*
    weight of the stencil:

        d~(0) = d(0) - d(-1)*(h - delta)/delta ,

    with delta the distance from the cell centre to the wall. Corrections from
    different directions are additive, and no ghost value is ever stored.

    Must be called on a stencil holding *only* the laplacian (right after
    fvm_lap), because it reads the off-diagonal laplacian weights d(-1).
    """
    for ax, shift, axis, h, sx, sy in [(1, -1, 1, dx,  1,  0),     # East
                                       (2,  1, 1, dx, -1,  0),     # West
                                       (3, -1, 0, dy,  0,  1),     # North
                                       (4,  1, 0, dy,  0, -1)]:    # South
        cut = fluid & np.roll(solid, shift, axis)
        if not cut.any():
            continue
        delta = wallDistance(sx, sy, XP, YP, h)
        a[0]  = np.where(cut, a[0] + a[ax]*(1.0-h/delta), a[0])
        a[ax] = np.where(cut, 0.0, a[ax])


def ibm_blankSolid(a):
    """
    Enforce u = 0 in cells whose centre lies inside the body.

    Applied to the *assembled* momentum stencil, so that the blanked diagonal
    keeps the same (positive) sign as every fluid diagonal. Folding it into the
    laplacian instead would leave -IBM_SOLID_WEIGHT on the diagonal, because the
    laplacian is subtracted, and hence a negative rAU / rAtU -- a negative
    diffusivity in the pressure equation -- inside the solid.
    """
    a[0] += np.where(solid, IBM_SOLID_WEIGHT, 0.0)


def ibmFaceWeights():
    """
    Per-face weights cLo, cHi such that

        phi_f = 0.5*(cLo*u_lo + cHi*u_hi)*|Sf|

    reproduces, on a face separating fluid from solid, the linear profile that
    vanishes on the true wall -- the collocated-grid counterpart of the linear
    extrapolation in ibm_lap. Away from the body both weights are 1 and this is
    the usual central interpolation. The geometry is static, so the bisection
    runs once here rather than every iteration.
    """
    def weights(axis, h, sx, sy, wrap):
        lo  = wrap(np.roll(solid, 1, axis))            # "west"/"south" cell of the face
        hi  = wrap(solid)                              # "east"/"north" cell of the face
        Xlo = wrap(np.roll(XP, 1, axis))
        Ylo = wrap(np.roll(YP, 1, axis))
        delta = wallDistance(sx, sy, Xlo, Ylo, h)      # from the lo cell towards hi
        cLo, cHi = np.ones_like(delta), np.ones_like(delta)
        m = lo & ~hi                                   # lo in the solid, hi in the fluid
        cLo[m], cHi[m] = 0.0, 1.0 - delta[m]/(h-delta[m])
        m = hi & ~lo                                   # hi in the solid, lo in the fluid
        cLo[m], cHi[m] = 2.0 - h/delta[m], 0.0
        return cLo, cHi
    return (weights(1, dx, 1, 0, wrapX),               # x faces
            weights(0, dy, 0, 1, wrapY))               # y faces


(cWx, cEx), (cSy, cNy) = ibmFaceWeights()


def interpolatePhiIBM(Ux, Uy):
    """Face volume fluxes from the cell-centred velocity, with the IBM weights."""
    return (0.5*(cWx*wrapX(np.roll(Ux, 1, 1)) + cEx*wrapX(Ux))*dy,
            0.5*(cSy*wrapY(np.roll(Uy, 1, 0)) + cNy*wrapY(Uy))*dx)

In [19]:
# Linear solvers
# --------------
# Both classes below only ever *precondition*. The matrices that are actually
# solved -- and therefore pEqn.flux() and the whole SIMPLEC splitting -- are
# untouched, so the algorithm stays exactly the OpenFOAM one.

class ReusedLU:
    """
    A SuperLU factorisation kept as a preconditioner across SIMPLE iterations.

    The momentum matrix changes a little every iteration (through the convective
    flux and dt), so the factorisation goes stale -- but a stale LU is still an
    excellent preconditioner, and the Krylov solve remains exact. Refactorise only
    when the iteration count says the old one has decayed. Because the LU is never
    used as the solver, the answer cannot depend on how stale it is.
    """

    def __init__(self, krylov, maxIts=20, rtol=1e-12):
        self.krylov, self.maxIts, self.rtol = krylov, maxIts, rtol
        self.lu, self.nFactor, self.nSolve, self.nIts = None, 0, 0, 0

    def solve(self, A, b, x0=None):
        if self.lu is None:
            self.lu = splu(A.tocsc()); self.nFactor += 1
        for attempt in range(2):
            its = [0]
            x, info = self.krylov(
                A, b, x0=x0, rtol=self.rtol, maxiter=500,
                M=LinearOperator(A.shape, matvec=self.lu.solve),
                callback=lambda xk: its.__setitem__(0, its[0]+1))
            if info == 0 and its[0] <= self.maxIts:
                break
            if attempt == 0:                                   # stale: refresh, retry
                self.lu = splu(A.tocsc()); self.nFactor += 1
        assert info == 0, f'momentum {self.krylov.__name__} failed, info={info}'
        self.nSolve += 1
        self.nIts += its[0]
        return x


class FftPoissonPC:
    """
    Preconditioner for the pressure equation: the *constant*-coefficient periodic
    Poisson operator with the same 5-point stencil, inverted exactly by FFT in
    O(N log N), wrapped in a symmetric Jacobi scaling.

    The Jacobi scaling is what makes this work at all: without it the pressure
    matrix has condition number ~1e17, because rAtU is ~1e-12 inside the solid
    (the IBM penalty) against ~1e2 in the fluid. That is a pure row/column scaling
    artefact -- symmetric Jacobi scaling alone brings it to ~5e4 -- and the FFT
    then removes the smooth error that Jacobi leaves behind.
    """

    def __init__(self):
        tx = 2*np.pi*np.fft.fftfreq(nx)
        ty = 2*np.pi*np.fft.fftfreq(ny)
        TX, TY = np.meshgrid(tx, ty)
        self.symbol = -2*((dy/dx)*(np.cos(TX)-1) + (dx/dy)*(np.cos(TY)-1))
        self.symbol[0, 0] = self.symbol[self.symbol > 0].min()   # keep it SPD

    def update(self, A, GammaBar):
        self.lam = GammaBar*self.symbol
        self.s   = 1.0/np.sqrt(np.abs(A.diagonal()))

    def matvec(self, r):
        z = self.s*r
        z = np.real(np.fft.ifft2(np.fft.fft2(z.reshape(ny, nx))/self.lam)).ravel()
        return self.s*z


def solvePressure(A, b, x0, pc):
    """CG on the SPD system (-A)p = (-b); fvm_lap has a negative diagonal."""
    its = [0]
    p, info = cg(-A, -b, x0=x0, rtol=pTol, maxiter=2000,
                 M=LinearOperator(A.shape, matvec=pc.matvec),
                 callback=lambda xk: its.__setitem__(0, its[0]+1))
    assert info == 0, f'pressure CG failed, info={info}'
    return p, its[0]

In [25]:
# Run controls
# ------------
startNew    = True
readRestart = False
consistent  = True             # True: SIMPLEC, False: SIMPLE

nIterations = 1000
convergenceTolerance = 1e-7   # stop the iterations when solution changes less than this
cflMax      = 100.0            # target Courant number of the pseudo-transient relaxation
dtMax       = 10.0


# Driving body force
alpha  = 0*np.pi/180
source = np.stack([np.where(fluid, 0.01*np.cos(alpha)*V, 0.0),
                   np.where(fluid, 0.01*np.sin(alpha)*V, 0.0)])

# Initial condition
if readRestart and not startNew:
    try:
        VAR      = np.fromfile('var.fld').reshape(VAR.shape)
        phiHbyAx = np.fromfile('phiHbyAx.fld').reshape((ny, nx+1))
        phiHbyAy = np.fromfile('phiHbyAy.fld').reshape((ny+1, nx))
    except (FileNotFoundError, ValueError):
        print('No restart field found, starting from rest.')
        startNew = True
if startNew:
    VAR[...] = 0.0
    phiHbyAx, phiHbyAy = interpolatePhiIBM(VAR[IU], VAR[IV])

# Static part of the system: geometry and viscosity do not change
lapl = zeroStencil()
fvm_lap(lapl, nu*np.ones((ny, nx)))
ibm_lap(lapl)

# Pressure reference, pinned in a *fluid* cell: inside the body the pressure
# equation only couples to the fluid through O(1/IBM_SOLID_WEIGHT) coefficients.
jRef, iRef = divmod(int(np.flatnonzero(fluid.ravel())[0]), nx)

uSolver = ReusedLU(bicgstab)
pPC     = FftPoissonPC()
t       = 0.0

# Timeloop
for it in range(nIterations):

    VARold = VAR.copy()

    # Assemble the momentum equation
    # ------------------------------
    a   = zeroStencil()
    rhs = np.zeros((2, ny, nx))

    fvm_ddt_LHS(a, dt)                                # d/dt
    a -= lapl                                         # -laplacian(nu, u), IBM-corrected
    fvm_divConvFlux(a, phiHbyAx, phiHbyAy)            # div(phi u)
    ibm_blankSolid(a)                                 # u = 0 inside the body

    for iC in range(2):
        fvm_ddt_RHS(rhs[iC], VAR[iC], dt)
    RHS   = (rhs+source).reshape(2, -1)
    gradP = fvc_grd(VAR[IP]).reshape(2, -1)

    # Momentum predictor
    # ------------------
    A = toCSR(a)
    if linearSolver == 'direct':
        lu = splu(A.tocsc())                          # one factorisation, two solves
        for iC in range(2):
            VAR[iC] = lu.solve(RHS[iC]-gradP[iC]).reshape(ny, nx)
    else:
        for iC in range(2):
            VAR[iC] = uSolver.solve(A, RHS[iC]-gradP[iC],
                                    x0=VAR[iC].ravel().copy()).reshape(ny, nx)

    # Split as  aP*u = H(u) - V*grad(p),  H(u) = source - offDiag*u
    # -------------------------------------------------------------
    aP     = a[0]                                     # OpenFOAM UEqn.A(), times V
    rAU    = 1.0/aP
    sumOff = a[1]+a[2]+a[3]+a[4]                      # OpenFOAM -UEqn.H1(), times V

    HbyA = np.stack([(RHS[iC].reshape(ny, nx) - offDiagDot(a, VAR[iC]))*rAU
                     for iC in range(2)])
    phiHbyAx, phiHbyAy = interpolatePhiIBM(HbyA[IU], HbyA[IV])
    meanDiv0 = np.mean(np.abs(fvc_divFlux(phiHbyAx, phiHbyAy)))

    # SIMPLEC consistency terms
    # -------------------------
    rAtU = rAU
    if consistent:
        rAtU = 1.0/(aP+sumOff)
        pFluxX, pFluxY = fvc_grdFlux(VAR[IP], (rAtU-rAU)*V)
        phiHbyAx += pFluxX
        phiHbyAy += pFluxY
        HbyA -= (rAU-rAtU)*gradP.reshape(2, ny, nx)

    # Pressure equation:  laplacian(rAtU, p) = div(phiHbyA)
    # -----------------------------------------------------
    rAtUV = rAtU*V
    pStencil = zeroStencil()
    fvm_lap(pStencil, rAtUV)
    pRHS = fvc_divFlux(phiHbyAx, phiHbyAy)
    pStencil[0, jRef, iRef] *= 2.0                    # OpenFOAM fvMatrix::setReference:
    Ap = toCSR(pStencil)                              # symmetric, unlike replacing the row
    if linearSolver == 'direct':
        VAR[IP] = splu(Ap.tocsc()).solve(pRHS.ravel()).reshape(ny, nx)
        pIts = 0
    else:
        pPC.update(Ap, rAtUV[fluid].mean())
        p, pIts = solvePressure(Ap, pRHS.ravel(), VAR[IP].ravel().copy(), pPC)
        VAR[IP] = p.reshape(ny, nx)

    # Correct the flux and the velocity
    # ---------------------------------
    pFluxX, pFluxY = fvc_grdFlux(VAR[IP], rAtUV)
    phiHbyAx -= pFluxX
    phiHbyAy -= pFluxY

    gradP = fvc_grd(VAR[IP])
    VAR[0:2] = HbyA - rAtU*gradP

    t += dt

    # Diagnostics
    # -----------
    divU    = fvc_divFlux(phiHbyAx, phiHbyAy)
    invTime = max(np.max(np.abs(VAR[IU]))/dx, np.max(np.abs(VAR[IV]))/dy)

    print(f'Iteration: {it}   t = {t:.3f}')
    print(f'------------------')
    print(f'                     Courant number of this step: {dt*invTime:.2f}')
    print(f'                               timestep used, dt: {dt:.3f}')
    print(f'    global continuity error: {np.sum(divU):.3e}   (sum of div(flux))')
    print(f'     final local continuity error: {np.mean(np.abs(divU)):.3e}')
    print(f'   initial local continuity error: {meanDiv0:.3e}')
    print(f'       maximal velocity in field: {np.max(np.sqrt(VAR[IU]**2+VAR[IV]**2)):.6f}')
    print(f'       maximal velocity in solid: {np.max(np.abs(VAR[0:2, solid])):.3e}')
    maxSolChange = np.max(np.abs(VARold[0:2]-VAR[0:2]))
    print(f'         maximum solution change: {maxSolChange:.6e}')
    if linearSolver != 'direct':
        print(f'      linear solver: {pIts} CG its (p), '
              f'{uSolver.nIts/max(uSolver.nSolve, 1):.1f} BiCGStab its/solve (U), '
              f'{uSolver.nFactor} factorisations so far')
    print(f'')

    # Timestep for the next iteration
    dt = dtMax if invTime == 0.0 else min(cflMax/invTime, dtMax)

    # Check convergence
    if maxSolChange<convergenceTolerance:
        print('Stopped due to reached solution change tolerance!')
        break

# Save solution
VAR.tofile('var.fld')
phiHbyAx.tofile('phiHbyAx.fld')
phiHbyAy.tofile('phiHbyAy.fld')

Iteration: 0   t = 3.966
------------------
                     Courant number of this step: 2.61
                               timestep used, dt: 3.966
    global continuity error: -7.293e-19   (sum of div(flux))
     final local continuity error: 3.090e-18
   initial local continuity error: 1.366e-05
       maximal velocity in field: 0.051444
       maximal velocity in solid: 1.711e-15
         maximum solution change: 5.143452e-02

Iteration: 1   t = 13.966
------------------
                     Courant number of this step: 20.66
                               timestep used, dt: 10.000
    global continuity error: -5.765e-19   (sum of div(flux))
     final local continuity error: 1.616e-17
   initial local continuity error: 2.525e-05
       maximal velocity in field: 0.161451
       maximal velocity in solid: 3.629e-15
         maximum solution change: 1.143922e-01

Iteration: 2   t = 23.966
------------------
                     Courant number of this step: 33.94
              

In [22]:
plt.figure()
plt.title('Volumes (green), fluxes (black & red) and pressure (colormap)')
xmin, xmax = Lx/2-1.3*R, Lx/2+1.3*R
ymin, ymax = Ly/2-1.3*R, Ly/2+1.3*R

theta = np.linspace(0, 2*np.pi, 100)
xBody = Lx/2 + R*np.cos(theta)
yBody = Ly/2 + R*np.sin(theta)

plt.plot(xBody, yBody, 'y')
plt.pcolormesh(XP, YP, VAR[IP])
for x in xf:
    plt.plot([x, x], [yf[0], yf[-1]], 'g', lw=0.5)
for y in yf:
    plt.plot([xf[0], xf[-1]], [y, y], 'g', lw=0.5)
plt.quiver(xf, yp, phiHbyAx, np.zeros_like(phiHbyAx), pivot='mid', scale=0.5)
plt.quiver(xp, yf, np.zeros_like(phiHbyAy), phiHbyAy, color='r', pivot='mid', scale=0.5)
for ix, x in enumerate(xp):
    for iy, y in enumerate(yp):
        if xmin < x < xmax and ymin < y < ymax:
            plt.text(x, y, f'{divU[iy, ix]:.1E}', horizontalalignment='center')
plt.colorbar()
plt.xlim([xmin, xmax])
plt.ylim([ymin, ymax])

(3.7, 6.3)

In [23]:
plt.figure()
plt.title('Colormap: p, vectors: u')
plt.pcolormesh(XP, YP, VAR[IP])
plt.colorbar()
plt.quiver(XP, YP, VAR[IU], VAR[IV])
plt.plot(xBody, yBody, 'y')

plt.figure()
plt.title('Colormap: u')
plt.pcolormesh(XP, YP, VAR[IU])
plt.colorbar()
plt.plot(xBody, yBody, 'y')

plt.figure()
plt.title(f'u (red/magenta) and p (blue/black) along y = {yp[ny//2]:.3f}; body centre is at {Ly/2:.3f}')
inBody = np.abs(xp-Lx/2) <= R
plt.plot(xp[~inBody], VAR[IU, ny//2, ~inBody], 'r+', label='u, fluid')
plt.plot(xp[inBody],  VAR[IU, ny//2,  inBody], 'm+', label='u, solid')
plt.plot(xp[~inBody], VAR[IP, ny//2, ~inBody], 'b+', label='p, fluid')
plt.plot(xp[inBody],  VAR[IP, ny//2,  inBody], 'k+', label='p, solid')
plt.legend()

plt.figure()
plt.title('Colormap: div(phi)')
plt.pcolormesh(XP, YP, divU)
plt.colorbar()